In [4]:
import os
import glob
import sys
from multiprocessing import Pool
from functools import partial

import cv2
from ultralytics import YOLO
from tqdm import tqdm
import torch

sys.path.append("../../PG_PyTraining")

import simber
from common.pg.cam_params import get_camparams
from ml_training.detectors.data_loaders import extract_height_from_filename
from benchmark_admin.bmark_files_factory import _num_of_frames
from common.pg import pg_geometry

In [5]:
LIINE_TEMPLATE = \
    "frame {frame_n} PG_OCSManagerImpl::PrintOneTrackToPos TRACK_USER_{IDX} {l} {t} {w} {h}  "\
    "POSITION_{x_cm}*{y_cm} STATUS_STANDING HEIGHT_0 ORIGINALPOSITION_{x}*{y}"

In [9]:
class Processor:
    def __init__(self, model_path, conf, iou, device, output_root_dir):
        self.model_path = model_path
        self.output_root_dir = output_root_dir
        
        self.args = dict(
            task="pose",
            device=[device],
            batch=128,
            verbose=False,
            conf=conf,
            iou=iou,
            imgsz=320,
            model="val",
            rect=False,
            single_cls=False,
            stream=True,
        )
    
    @staticmethod
    def flip_video(in_path, out_path, flip_code):
        # Open the video file
        cap = cv2.VideoCapture(in_path)

        # Get video properties
        fps = cap.get(cv2.CAP_PROP_FPS)
        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        # Define the codec and create VideoWriter object
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # For .mp4 files
        out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # Flip the frame
            flipped_frame = cv2.flip(frame, flip_code)

            # Write the flipped frame to output
            out.write(flipped_frame)

        # Release everything
        cap.release()
        out.release()
    
    def process_video(self, video_path):
        os.makedirs(self.output_root_dir, exist_ok=True)
        output_path = os.path.basename(video_path)

        model = YOLO(self.model_path)

        cam_params = get_camparams(video_path)
        cam_height = extract_height_from_filename(video_path)
        cam = pg_geometry.PgCamera(cam_height * 10, cam_params.NAME)

        self.flip_video(video_path, output_path, cam_params.FLIP_CODE)

        lines = []
        frame_count = 0
        for i in model.predict(output_path, **self.args):
            xywhs = i.boxes.xywh.detach().cpu().numpy()
            probs = i.boxes.conf.detach().cpu().numpy()
            xys = i.keypoints.xy.detach().cpu().numpy()
            
            for idx, (xywh, prob, xy) in enumerate(zip(xywhs, probs, xys)):
                l, t, w, h = xywh
                x, y = xy[0]
                x_cm, y_cm = cam.pixel2cm([[x, y]])[0]
                lines.append(
                    LIINE_TEMPLATE.format(
                        frame_n=frame_count,
                        IDX=idx,
                        l=int(l - w / 2),
                        t=int(t - h / 2),
                        w=int(w),
                        h=int(h),
                        x_cm=int(x_cm),
                        y_cm=int(y_cm),
                        x=int(x),
                        y=int(y)
                    )
                )
            frame_count += 1

        torch.cuda.empty_cache()
        os.remove(output_path)
        
        header_lines = [
            video_path,
            f";META ceiling_height_cm {cam_height}",
            "frame 0 PG_MainThread::Run START",
        ]

        with open(os.path.join(self.output_root_dir, output_path.replace('.avi', '.pos')), "w") as f:
            f.write("\n".join(header_lines + lines + [f"frame {frame_count} PG_MainThread::Run END"]))

In [ ]:
def process_video(video_path, res_folder):
    output_path = os.path.basename(video_path)

    # if os.path.exists(f"{res_folder}/{output_path.replace('.avi', '.pos')}"):
    #     return

    model = YOLO("../people_detection_pg2/results/filtered_v3_margin_15/weights/best.pt")

    cam_params = get_camparams(video_path)
    cam_height = extract_height_from_filename(video_path)
    num_frames = _num_of_frames(video_path)
    
    cam = pg_geometry.PgCamera(cam_height * 10, cam_params.NAME)
    
    # Open the video file
    cap = cv2.VideoCapture(video_path)

    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # For .mp4 files
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Flip type: 0=vertical, 1=horizontal, -1=both
    flip_code = 1

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Flip the frame
        flipped_frame = cv2.flip(frame, cam_params.FLIP_CODE)

        # Write the flipped frame to output
        out.write(flipped_frame)

    # Release everything
    cap.release()
    out.release()
    
    args = dict(
        task="pose",
        device=[1],
        batch=128,
        verbose=False,
        conf=0.35,
        iou=0.5,
        imgsz=320,
        model="val",
        rect=False,
        single_cls=False,
        stream=True,
    )

    lines = []
    frame_count = 0
    for i in model.predict(output_path, **args):
        xywhs = i.boxes.xywh.detach().cpu().numpy()
        probs = i.boxes.conf.detach().cpu().numpy()
        xys = i.keypoints.xy.detach().cpu().numpy()
        
        for idx, (xywh, prob, xy) in enumerate(zip(xywhs, probs, xys)):
            l, t, w, h = xywh
            x, y = xy[0]
            x_cm, y_cm = cam.pixel2cm([[x, y]])[0]
            lines.append(
                LIINE_TEMPLATE.format(
                    frame_n=frame_count,
                    IDX=idx,
                    l=int(l - w / 2),
                    t=int(t - h / 2),
                    w=int(w),
                    h=int(h),
                    x_cm=int(x_cm),
                    y_cm=int(y_cm),
                    x=int(x),
                    y=int(y)
                )
            )
        frame_count += 1

    os.remove(output_path)
    
    header_lines = [
        video_path,
        f";META ceiling_height_cm {cam_height}",
        "frame 0 PG_MainThread::Run START",
    ]

    os.makedirs(res_folder, exist_ok=True)
    with open(f"{res_folder}/{output_path.replace('.avi', '.pos')}", "w") as f:
        f.write("\n".join(header_lines + lines + [f"frame {frame_count} PG_MainThread::Run END"]))
    
    torch.cuda.empty_cache()


In [4]:
# /mnt/pg-fs01/Current/Benchmarks/PG1_AlgoDev_DLtest02_RV9,
# /mnt/pg-fs01/Current/Benchmarks/PG1_AlgoDev_DLtest_RV59,
# /mnt/pg-fs01/Current/Benchmarks/PG1_AlgoDev_VLC_RV5_V2,
# /mnt/pg-fs01/Current/DB/CV_PeopleTracking/Benchmarks/Mixed/VLC_MultiplePeopleCrossing/10fps

In [5]:
seq_paths = glob.glob("/mnt/pg-fs01/Current/Benchmarks/PG1_AlgoDev_VLC_RV5_V2/*.seq")
video_paths = simber.videos_from_seq_files(seq_paths)
print(len(video_paths))

f = partial(process_video, res_folder="PG1_AlgoDev_VLC_RV5_V2")

161


In [6]:
with Pool(4) as p:
    r = list(tqdm(p.imap(f, video_paths), total=len(video_paths)))

  0%|          | 0/161 [00:00<?, ?it/s]

100%|██████████| 161/161 [26:49<00:00, 10.00s/it]


In [11]:
seq_paths = glob.glob("/mnt/pg-fs01/Current/DB/CV_PeopleTracking/Benchmarks/Mixed/VLC_MultiplePeopleCrossing/10fps/*.seq")
video_paths = simber.videos_from_seq_files(seq_paths)
print(len(video_paths))

processor = Processor(
    model_path="../people_detection_pg2/results/filtered_v3_margin_15/weights/best.pt",
    conf=0.35,
    iou=0.5,
    device=1,
    output_root_dir="test_run/10fps"
)

with Pool(4) as p:
    r = list(tqdm(p.imap(processor.process_video, video_paths), total=len(video_paths)))

16


100%|██████████| 16/16 [01:06<00:00,  4.18s/it]
